In [1]:
%pip install -Uqq transformers optuna catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 18.9 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 625, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 625 (delta 146), reused 86 (delta 85), pack-reused 434 (from 3)
Receiving objects: 100% (625/625), 206.72 KiB | 1.52 MiB/s, done.
Resolving deltas: 100% (320/320), done.


# Import Libraries

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier
import optuna
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
from cuml.linear_model import LogisticRegression

# Read Data

In [2]:
# Load the SAME splits you used for fine-tuned BERT
train_df = pd.read_csv('/content/train_split.csv')
test_df = pd.read_csv('/content/test_split.csv')

# Verify it's the same size
print(f"Train: {len(train_df):,} samples")
print(f"Test: {len(test_df):,} samples")

# Extract texts and labels
train_texts = train_df['text'].tolist()
train_labels = train_df['rate'].values

test_texts = test_df['text'].tolist()
test_labels = test_df['rate'].values

print(f"Train: {len(train_texts):,} samples")
print(f"Test: {len(test_texts):,} samples")


Train: 42,456 samples
Test: 4,718 samples
Train: 42,456 samples
Test: 4,718 samples


In [3]:
train_df['rate'].value_counts(normalize=True)

,proportion
rate,
5,0.525791
4,0.206967
3,0.128674
1,0.087549
2,0.051018


# Load Bert and Extract Features for the Classificator

In [4]:
ru_bert = "DeepPavlov/rubert-base-cased-conversational"

tokenizer = AutoTokenizer.from_pretrained(ru_bert)
model = AutoModel.from_pretrained(ru_bert)

text = " ".join(["Очень понравилось. Были в начале марта ! "])
tokenized_text = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    output = model(**tokenized_text)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased-conversational
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
output.last_hidden_state.shape

torch.Size([1, 10, 768])

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def process_in_batches(text_list, tokenizer, model, batch_size=16):
    """
    Process text data in batches to avoid memory issues
    """
    all_features = []

    with torch.no_grad():
        for i in range(0, len(text_list), batch_size):
            batch_texts = text_list[i:i+batch_size]
            batch_texts = [str(text) for text in batch_texts]
            tokenized = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512,
                return_attention_mask=True
            )
            input_ids = tokenized["input_ids"].to(device)
            attention_mask = tokenized["attention_mask"].to(device)

            # === ADD THIS ONE LINE FOR MIXED PRECISION ===
            with torch.amp.autocast('cuda'):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # =============================================

            # Use [CLS] token representation (first token)
            batch_features = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_features.append(batch_features)

            # Clear memory
            del input_ids, attention_mask, outputs, batch_features, tokenized
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Concatenate all batches
    features = np.concatenate(all_features, axis=0)
    print(f"✅ Completed! Features shape: {features.shape}")
    return features

In [8]:
train_features = process_in_batches(train_texts, tokenizer, model, batch_size=16)

✅ Completed! Features shape: (42456, 768)


# Initialize Logistic Regression and Run Cross Validation using GPU

In [9]:
lr_cuml = LogisticRegression(
    penalty="l2",
    max_iter=2000
)

In [11]:
# Cross-validation
cv = 5
skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

train_f1_scores = []
val_f1_scores = []

train_texts = train_features

# Encode labels once outside the loop for consistency
le = LabelEncoder()
labels_encoded = le.fit_transform(train_labels)

for fold, (train_idx, val_idx) in enumerate(tqdm(skf.split(train_texts, labels_encoded), total=cv, desc="CV Progress")):
    fold_train_texts = train_texts[train_idx]
    fold_val_texts = train_texts[val_idx]

    y_train_encoded = labels_encoded[train_idx]
    y_val_encoded = labels_encoded[val_idx]
    y_train_original = le.inverse_transform(y_train_encoded)
    y_val_original = le.inverse_transform(y_val_encoded)

    # Train
    lr_cuml.fit(fold_train_texts, y_train_encoded)

    train_preds_proba = lr_cuml.predict_proba(fold_train_texts)
    val_preds_proba = lr_cuml.predict_proba(fold_val_texts)

    train_preds_encoded = np.argmax(train_preds_proba, axis=1)
    val_preds_encoded = np.argmax(val_preds_proba, axis=1)

    train_preds_original = le.inverse_transform(train_preds_encoded)
    val_preds_original = le.inverse_transform(val_preds_encoded)

    # Calculate F1 for both train and validation
    train_f1 = f1_score(y_train_original, train_preds_original, average='weighted', zero_division=0)
    val_f1 = f1_score(y_val_original, val_preds_original, average='weighted', zero_division=0)
    train_f1_scores.append(train_f1)
    val_f1_scores.append(val_f1)

    print(f"Fold {fold + 1}:")
    print(f"  Train F1: {train_f1:.4f}, Val F1: {val_f1:.4f}, Gap: {train_f1 - val_f1:.4f}")
    print("-" * 50)

metrics_data = []
for i in range(cv):
    metrics_data.append({
        'fold': i + 1,
        'train_f1': train_f1_scores[i],
        'val_f1': val_f1_scores[i]
    })
metrics_df = pd.DataFrame(metrics_data)

# Results
print("\n" + "=" * 60)
print("FINAL SUMMARY - 5-FOLD CROSS VALIDATION")
print("=" * 60)
print(f"Train F1: {metrics_df['train_f1'].mean():.4f} (+/- {metrics_df['train_f1'].std():.4f})")
print(f"Val F1:   {metrics_df['val_f1'].mean():.4f} (+/- {metrics_df['val_f1'].std():.4f})")
print(f"F1 Gap:   {metrics_df['train_f1'].mean() - metrics_df['val_f1'].mean():.4f}")

print(f"\nIndividual Fold Validation F1 Scores:")
for i, f1 in enumerate(val_f1_scores, 1):
    print(f"  Fold {i}: {f1:.4f}")

CV Progress:  20%|██        | 1/5 [00:03<00:15,  3.96s/it]

Fold 1:
  Train F1: 0.6586, Val F1: 0.6149, Gap: 0.0437
--------------------------------------------------


CV Progress:  40%|████      | 2/5 [00:05<00:07,  2.67s/it]

Fold 2:
  Train F1: 0.6545, Val F1: 0.6173, Gap: 0.0373
--------------------------------------------------
[2026-02-02 08:26:22.135] [CUML] [warning] L-BFGS stopped, because the line search failed to advance (step delta = 0.000000)


CV Progress:  60%|██████    | 3/5 [00:07<00:04,  2.20s/it]

Fold 3:
  Train F1: 0.6557, Val F1: 0.6130, Gap: 0.0426
--------------------------------------------------


CV Progress:  80%|████████  | 4/5 [00:09<00:02,  2.06s/it]

Fold 4:
  Train F1: 0.6562, Val F1: 0.6156, Gap: 0.0406
--------------------------------------------------


CV Progress: 100%|██████████| 5/5 [00:10<00:00,  2.13s/it]

Fold 5:
  Train F1: 0.6561, Val F1: 0.6150, Gap: 0.0411
--------------------------------------------------

FINAL SUMMARY - 5-FOLD CROSS VALIDATION
Train F1: 0.6562 (+/- 0.0015)
Val F1:   0.6152 (+/- 0.0015)
F1 Gap:   0.0411

Individual Fold Validation F1 Scores:
  Fold 1: 0.6149
  Fold 2: 0.6173
  Fold 3: 0.6130
  Fold 4: 0.6156
  Fold 5: 0.6150


In [12]:
# Train final model on ALL training data

final_le = LabelEncoder()
all_train_labels_encoded = final_le.fit_transform(train_labels)

final_lr = LogisticRegression(
    penalty='l2',
    max_iter=2000,
)
final_lr.fit(train_features, all_train_labels_encoded)

# Extract features for test set
print("\nExtracting features for test set...")
test_features = process_in_batches(test_texts, tokenizer, model, batch_size=16)

# Make predictions on TRAINING data
train_preds_encoded = final_lr.predict(train_features)
train_preds_original = final_le.inverse_transform(train_preds_encoded)

# Make predictions on TEST data
test_labels_encoded = final_le.transform(test_labels)
test_preds_encoded = final_lr.predict(test_features)
test_preds_original = final_le.inverse_transform(test_preds_encoded)

# Calculate F1 scores
train_f1 = f1_score(train_labels, train_preds_original, average='weighted')
test_f1 = f1_score(test_labels, test_preds_original, average='weighted')

print(f"\n✅ Train set results (final model on ALL training data):")
print(f"   Samples: {len(train_texts):,}")
print(f"   F1 Score: {train_f1:.4f}")

print(f"\n✅ Test set results (held-out test data):")
print(f"   Samples: {len(test_texts):,}")
print(f"   F1 Score: {test_f1:.4f}")


Extracting features for test set...
✅ Completed! Features shape: (4718, 768)

✅ Train set results (final model on ALL training data):
   Samples: 42,456
   F1 Score: 0.6501

✅ Test set results (held-out test data):
   Samples: 4,718
   F1 Score: 0.6200


# Now let's run CatBoostClassifier with the same Bert features and optuna parameters

In [23]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'depth': trial.suggest_int('depth', 3, 5),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 5, 20),
        'random_strength': trial.suggest_float('random_strength', 1, 5),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.5, 1.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 20),
        'random_seed': 42,
        'task_type': 'GPU',
        'verbose': False,
        'eval_metric': 'MultiClass'
    }

    cv = 3
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    train_texts = train_features
    y = train_labels

    # Encode labels
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    train_f1_scores = []
    val_f1_scores = []

    for train_idx, val_idx in skf.split(train_texts, y_encoded):
        fold_train_texts = train_texts[train_idx]
        fold_val_texts = train_texts[val_idx]

        y_train_encoded = y_encoded[train_idx]
        y_val_encoded = y_encoded[val_idx]
        y_train_original = le.inverse_transform(y_train_encoded)
        y_val_original = le.inverse_transform(y_val_encoded)

        catboost_model = CatBoostClassifier(**params)
        catboost_model.fit(
            fold_train_texts, y_train_encoded,
            eval_set=(fold_val_texts, y_val_encoded),
            verbose=False,
            early_stopping_rounds=30
        )

        # Predictions
        train_preds_encoded = catboost_model.predict(fold_train_texts).flatten()
        val_preds_encoded = catboost_model.predict(fold_val_texts).flatten()

        train_preds_original = le.inverse_transform(train_preds_encoded)
        val_preds_original = le.inverse_transform(val_preds_encoded)

        train_f1 = f1_score(y_train_original, train_preds_original, average='weighted', zero_division=0)
        val_f1 = f1_score(y_val_original, val_preds_original, average='weighted', zero_division=0)

        train_f1_scores.append(train_f1)
        val_f1_scores.append(val_f1)

    mean_train_f1 = np.mean(train_f1_scores)
    mean_val_f1 = np.mean(val_f1_scores)
    overfitting_gap = mean_train_f1 - mean_val_f1

    # Penalize overfitting in the objective
    penalty = 0.1 * overfitting_gap if overfitting_gap > 0.02 else 0
    penalized_score = mean_val_f1 - penalty

    print(f"Val F1: {mean_val_f1:.4f}, Gap: {overfitting_gap:.4f}, Penalized: {penalized_score:.4f}")

    return penalized_score

# Run Optuna optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print("Best trial:")
trial = study.best_trial
print(f"  Value (F1 Score): {trial.value:.4f}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

# Use best parameters for final training
best_params = study.best_params
best_params.update({
    'random_seed': 42,
    'task_type': 'GPU',
    'eval_metric': 'MultiClass',
    'early_stopping_rounds': 20,
    'verbose': True
})

catboost = CatBoostClassifier(**best_params)

[I 2026-02-02 09:49:01,890] A new study created in memory with name: no-name-97901c4f-e55f-4952-89d8-ef0e34ea2763


  0%|          | 0/20 [00:00<?, ?it/s]

Val F1: 0.5622, Gap: 0.0142, Penalized: 0.5622
[I 2026-02-02 09:49:27,590] Trial 0 finished with value: 0.5622341415496215 and parameters: {'iterations': 300, 'learning_rate': 0.024375962138900364, 'depth': 4, 'l2_leaf_reg': 11.882156742428325, 'random_strength': 2.551876236437047, 'bagging_temperature': 0.6922440936733404, 'min_data_in_leaf': 12}. Best is trial 0 with value: 0.5622341415496215.
Val F1: 0.5680, Gap: 0.0123, Penalized: 0.5680
[I 2026-02-02 09:49:51,722] Trial 1 finished with value: 0.5680100034681576 and parameters: {'iterations': 422, 'learning_rate': 0.028423901155320466, 'depth': 3, 'l2_leaf_reg': 13.084569098838397, 'random_strength': 4.085767524373002, 'bagging_temperature': 0.9773063623727336, 'min_data_in_leaf': 17}. Best is trial 1 with value: 0.5680100034681576.
Val F1: 0.5897, Gap: 0.0399, Penalized: 0.5857
[I 2026-02-02 09:50:15,893] Trial 2 finished with value: 0.5857179673245094 and parameters: {'iterations': 302, 'learning_rate': 0.07895927131979738, 'dept

In [24]:
# Cross-validation
cv = 5
skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

train_f1_scores = []
val_f1_scores = []
metrics = []

# Use the same train split as other models
train_texts = train_features
labels = train_labels

# Encode labels once outside the loop for consistency
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

for fold, (train_idx, val_idx) in enumerate(tqdm(skf.split(train_texts, labels_encoded), total=cv, desc="CV Progress")):
    fold_train_texts = train_texts[train_idx]
    fold_val_texts = train_texts[val_idx]

    y_train_encoded = labels_encoded[train_idx]
    y_val_encoded = labels_encoded[val_idx]
    y_train_original = le.inverse_transform(y_train_encoded)
    y_val_original = le.inverse_transform(y_val_encoded)

    # Train CatBoost
    catboost.fit(fold_train_texts, y_train_encoded)

    # Get predictions
    train_preds_encoded = catboost.predict(fold_train_texts).flatten()
    val_preds_encoded = catboost.predict(fold_val_texts).flatten()

    train_preds_original = le.inverse_transform(train_preds_encoded)
    val_preds_original = le.inverse_transform(val_preds_encoded)

    # Calculate F1 for both train and validation
    train_f1 = f1_score(y_train_original, train_preds_original, average='weighted', zero_division=0)
    val_f1 = f1_score(y_val_original, val_preds_original, average='weighted', zero_division=0)
    train_f1_scores.append(train_f1)
    val_f1_scores.append(val_f1)

    metrics.append({
        'fold': fold + 1,
        'train_f1': train_f1,
        'val_f1': val_f1
    })

    print(f"Fold {fold + 1}:")
    print(f"  Train F1: {train_f1:.4f}, Val F1: {val_f1:.4f}, Gap: {train_f1 - val_f1:.4f}")
    print("-" * 50)

# Results
metrics_df = pd.DataFrame(metrics)
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"Train F1: {metrics_df['train_f1'].mean():.4f} (+/- {metrics_df['train_f1'].std():.4f})")
print(f"Val F1:   {metrics_df['val_f1'].mean():.4f} (+/- {metrics_df['val_f1'].std():.4f})")
print(f"F1 Gap:   {metrics_df['train_f1'].mean() - metrics_df['val_f1'].mean():.4f}")

CV Progress:   0%|          | 0/5 [00:00<?, ?it/s]

0:	learn: 1.5402593	total: 37ms	remaining: 18.5s
1:	learn: 1.4837607	total: 71.4ms	remaining: 17.8s
2:	learn: 1.4378880	total: 103ms	remaining: 17.1s
3:	learn: 1.3982645	total: 147ms	remaining: 18.2s
4:	learn: 1.3634451	total: 193ms	remaining: 19.1s
5:	learn: 1.3337401	total: 242ms	remaining: 20s
6:	learn: 1.3071374	total: 288ms	remaining: 20.3s
7:	learn: 1.2837081	total: 343ms	remaining: 21.1s
8:	learn: 1.2624763	total: 396ms	remaining: 21.6s
9:	learn: 1.2435709	total: 449ms	remaining: 22s
10:	learn: 1.2262072	total: 504ms	remaining: 22.4s
11:	learn: 1.2101519	total: 556ms	remaining: 22.6s
12:	learn: 1.1954804	total: 593ms	remaining: 22.2s
13:	learn: 1.1822759	total: 617ms	remaining: 21.4s
14:	learn: 1.1702530	total: 643ms	remaining: 20.8s
15:	learn: 1.1599052	total: 696ms	remaining: 21s
16:	learn: 1.1494827	total: 750ms	remaining: 21.3s
17:	learn: 1.1398503	total: 806ms	remaining: 21.6s
18:	learn: 1.1307143	total: 850ms	remaining: 21.5s
19:	learn: 1.1224349	total: 900ms	remaining: 21

CV Progress:  20%|██        | 1/5 [00:13<00:52, 13.10s/it]

Fold 1:
  Train F1: 0.6331, Val F1: 0.5934, Gap: 0.0397
--------------------------------------------------
0:	learn: 1.5400344	total: 38.4ms	remaining: 19.1s
1:	learn: 1.4838746	total: 67.1ms	remaining: 16.7s
2:	learn: 1.4368899	total: 111ms	remaining: 18.4s
3:	learn: 1.3972886	total: 162ms	remaining: 20.1s
4:	learn: 1.3627202	total: 212ms	remaining: 21s
5:	learn: 1.3334100	total: 259ms	remaining: 21.3s
6:	learn: 1.3071360	total: 308ms	remaining: 21.7s
7:	learn: 1.2837755	total: 355ms	remaining: 21.9s
8:	learn: 1.2627062	total: 389ms	remaining: 21.2s
9:	learn: 1.2436183	total: 425ms	remaining: 20.8s
10:	learn: 1.2265377	total: 452ms	remaining: 20.1s
11:	learn: 1.2115860	total: 492ms	remaining: 20s
12:	learn: 1.1970898	total: 517ms	remaining: 19.4s
13:	learn: 1.1839712	total: 551ms	remaining: 19.1s
14:	learn: 1.1718326	total: 591ms	remaining: 19.1s
15:	learn: 1.1605706	total: 627ms	remaining: 19s
16:	learn: 1.1505635	total: 649ms	remaining: 18.4s
17:	learn: 1.1406482	total: 670ms	remain

CV Progress:  40%|████      | 2/5 [00:23<00:35, 11.74s/it]

...

Fold 3:
  Train F1: 0.6324, Val F1: 0.5915, Gap: 0.0409
--------------------------------------------------
0:	learn: 1.5400160	total: 28.4ms	remaining: 14.2s
1:	learn: 1.4839887	total: 53.2ms	remaining: 13.2s
2:	learn: 1.4377683	total: 77.3ms	remaining: 12.8s
3:	learn: 1.3986213	total: 102ms	remaining: 12.7s
4:	learn: 1.3644080	total: 126ms	remaining: 12.5s
5:	learn: 1.3343366	total: 150ms	remaining: 12.4s
6:	learn: 1.3085001	total: 174ms	remaining: 12.3s
7:	learn: 1.2847331	total: 198ms	remaining: 12.2s
8:	learn: 1.2635909	total: 223ms	remaining: 12.2s
9:	learn: 1.2451224	total: 244ms	remaining: 12s
10:	learn: 1.2278954	total: 257ms	remaining: 11.4s
11:	learn: 1.2124704	total: 272ms	remaining: 11s
12:	learn: 1.1981798	total: 285ms	remaining: 10.7s
13:	learn: 1.1848216	total: 298ms	remaining: 10.4s
14:	learn: 1.1725534	total: 311ms	remaining: 10.1s
15:	learn: 1.1610995	total: 325ms	remaining: 9.82s
16:	learn: 1.1507853	total: 338ms	remaining: 9.59s
17:	learn: 1.1409445	total: 351ms	rem

CV Progress:  80%|████████  | 4/5 [00:45<00:11, 11.15s/it]

Fold 4:
  Train F1: 0.6315, Val F1: 0.6019, Gap: 0.0295
--------------------------------------------------
0:	learn: 1.5397879	total: 29ms	remaining: 14.5s
1:	learn: 1.4829982	total: 53.1ms	remaining: 13.2s
2:	learn: 1.4371002	total: 77ms	remaining: 12.7s
3:	learn: 1.3975353	total: 101ms	remaining: 12.5s
4:	learn: 1.3628208	total: 126ms	remaining: 12.4s
5:	learn: 1.3328122	total: 150ms	remaining: 12.3s
6:	learn: 1.3065382	total: 174ms	remaining: 12.2s
7:	learn: 1.2837086	total: 198ms	remaining: 12.1s
8:	learn: 1.2624004	total: 222ms	remaining: 12.1s
9:	learn: 1.2430328	total: 249ms	remaining: 12.2s
10:	learn: 1.2265480	total: 274ms	remaining: 12.2s
11:	learn: 1.2109082	total: 290ms	remaining: 11.8s
12:	learn: 1.1966672	total: 304ms	remaining: 11.4s
13:	learn: 1.1832630	total: 317ms	remaining: 11s
14:	learn: 1.1711730	total: 330ms	remaining: 10.7s
15:	learn: 1.1601015	total: 344ms	remaining: 10.4s
16:	learn: 1.1497118	total: 357ms	remaining: 10.1s
17:	learn: 1.1400972	total: 371ms	remai

CV Progress: 100%|██████████| 5/5 [00:56<00:00, 11.29s/it]

Fold 5:
  Train F1: 0.6322, Val F1: 0.5913, Gap: 0.0409
--------------------------------------------------

FINAL SUMMARY
Train F1: 0.6320 (+/- 0.0009)
Val F1:   0.5946 (+/- 0.0044)
F1 Gap:   0.0374


In [25]:
# Train final model on ALL training data
all_train_labels_encoded = le.fit_transform(train_labels)

final_catboost = CatBoostClassifier(**best_params)
final_catboost.fit(train_features, all_train_labels_encoded)

# Predict on training set (for train F1)
train_preds_encoded = final_catboost.predict(train_features).flatten()
train_preds_original = le.inverse_transform(train_preds_encoded)
train_f1 = f1_score(train_labels, train_preds_original, average='weighted', zero_division=0)

# Evaluate on test set
print("Evaluating on test set...")
test_features = process_in_batches(test_texts, tokenizer, model, batch_size=16)
test_labels_encoded = le.transform(test_labels)
test_preds_encoded = final_catboost.predict(test_features).flatten()
test_preds_original = le.inverse_transform(test_preds_encoded)

test_f1 = f1_score(test_labels, test_preds_original, average='weighted', zero_division=0)

print(f"\n✅ CatBoost Train F1: {train_f1:.4f}")
print(f"✅ CatBoost Test F1:  {test_f1:.4f}")
print(f"✅ Overfitting Gap:   {train_f1 - test_f1:.4f}")

0:	learn: 1.5395785	total: 30.6ms	remaining: 15.2s
1:	learn: 1.4835563	total: 58.3ms	remaining: 14.5s
2:	learn: 1.4368129	total: 84.5ms	remaining: 14s
3:	learn: 1.3971147	total: 111ms	remaining: 13.7s
4:	learn: 1.3633134	total: 137ms	remaining: 13.6s
5:	learn: 1.3332439	total: 163ms	remaining: 13.4s
6:	learn: 1.3064569	total: 189ms	remaining: 13.3s
7:	learn: 1.2832784	total: 215ms	remaining: 13.2s
8:	learn: 1.2624178	total: 242ms	remaining: 13.2s
9:	learn: 1.2437533	total: 257ms	remaining: 12.6s
10:	learn: 1.2270490	total: 271ms	remaining: 12.1s
11:	learn: 1.2112710	total: 285ms	remaining: 11.6s
12:	learn: 1.1965945	total: 299ms	remaining: 11.2s
13:	learn: 1.1835731	total: 313ms	remaining: 10.9s
14:	learn: 1.1711783	total: 327ms	remaining: 10.6s
15:	learn: 1.1598501	total: 341ms	remaining: 10.3s
16:	learn: 1.1495059	total: 355ms	remaining: 10.1s
17:	learn: 1.1400584	total: 369ms	remaining: 9.89s
18:	learn: 1.1311881	total: 383ms	remaining: 9.71s
19:	learn: 1.1226673	total: 397ms	remain